<a href="https://colab.research.google.com/github/halimAhtasham/Network-_Intrusion_Detection---All_In_One/blob/main/ROSIDS23_Leakage_Free_IDS_Framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

import os

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')
print("Ready")

Ready


In [2]:
from google.colab import drive

In [3]:
drive.mount("/content/drive")

Mounted at /content/drive


# **Load the Dataset**

In [4]:
data_path = '/content/drive/MyDrive/Dataset'
files = [file for file in os.listdir(data_path) if file.endswith('.csv')]

print(f"Found {len(files)} CSV files in {data_path}:")
for file in files:
    print(f"- {file}")

Found 6 CSV files in /content/drive/MyDrive/Dataset:
- DoS.csv
- normal.csv
- subscriberflood.csv
- unauthorizedpublisher.csv
- unauthorizedsubscriber.csv
- ROSIDS23.csv


In [5]:
files = {
    "ROSIDS23": "ROSIDS23.csv",
    "Normal": "normal.csv",
    "DoS": "DoS.csv",
    "SubscriberFlood": "subscriberflood.csv",
    "UnauthorizedSubscriber": "unauthorizedsubscriber.csv",
    "UnauthorizedPublisher": "unauthorizedpublisher.csv"
}

In [6]:
datasets = {}

for name, file in files.items():
    full_file_path = os.path.join(data_path, file)
    try:
        datasets[name] = pd.read_csv(full_file_path)
        print(name, datasets[name].shape)
    except FileNotFoundError:
        print(f"Error: File not found at {full_file_path}")
    except Exception as e:
        print(f"An error occurred while reading {full_file_path}: {e}")

ROSIDS23 (136681, 84)
Normal (21416, 84)
DoS (600015, 84)
SubscriberFlood (46599, 84)
UnauthorizedSubscriber (12991, 84)
UnauthorizedPublisher (14791, 84)


# **Dataset Analysis**

In [7]:
for name, data in datasets.items():
  print("\n", name, ":")
  print(data.columns.tolist())


 ROSIDS23 :
['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Fwd By

In [8]:
for name, data in datasets.items():
  print("\n", name)
  print(data.dtypes.value_counts())


 ROSIDS23
float64    45
int64      34
object      5
Name: count, dtype: int64

 Normal
float64    45
int64      34
object      5
Name: count, dtype: int64

 DoS
float64    45
int64      35
object      4
Name: count, dtype: int64

 SubscriberFlood
float64    45
int64      35
object      4
Name: count, dtype: int64

 UnauthorizedSubscriber
float64    45
int64      35
object      4
Name: count, dtype: int64

 UnauthorizedPublisher
float64    45
int64      35
object      4
Name: count, dtype: int64


In [9]:
for name, data in datasets.items():
  print("\n",name)
  print(data['Label'].value_counts())
  print(data['Label'].unique())


 ROSIDS23
Label
Benign       62511
DoS          31000
Subflood     30064
UnauthPub     7817
UnauthSub     5289
Name: count, dtype: int64
['Benign' 'DoS' 'Subflood' 'UnauthPub' 'UnauthSub']

 Normal
Label
No Label    21416
Name: count, dtype: int64
['No Label']

 DoS
Label
1    590131
0      9884
Name: count, dtype: int64
[0 1]

 SubscriberFlood
Label
1    30064
0    16535
Name: count, dtype: int64
[0 1]

 UnauthorizedSubscriber
Label
0    7702
1    5289
Name: count, dtype: int64
[0 1]

 UnauthorizedPublisher
Label
1    7817
0    6974
Name: count, dtype: int64
[0 1]


In [10]:
import hashlib

def dataframe_hash(df):
    return pd.util.hash_pandas_object(
        df,
        index=False
    )

ros_hash = set(
    dataframe_hash(datasets["ROSIDS23"])
)


for name in ["Normal",
             "DoS",
             "SubscriberFlood",
             "UnauthorizedSubscriber",
             "UnauthorizedPublisher"]:

    temp_hash = dataframe_hash(datasets[name])

    overlap = temp_hash.isin(ros_hash).sum()

    print(name, "overlap:", overlap)

Normal overlap: 0
DoS overlap: 0
SubscriberFlood overlap: 0
UnauthorizedSubscriber overlap: 0
UnauthorizedPublisher overlap: 0


# **Dataset Profiling**

In [11]:
ros = datasets["ROSIDS23"]

for col in [
    "Flow ID",
    "Src IP",
    "Dst IP",
    "Protocol",
    "Timestamp",
    "Src Port",
    "Dst Port"
]:
    print("\n")
    print(col)
    print("Unique values:", ros[col].nunique())
    print(ros[col].head())



Flow ID
Unique values: 71793
0    192.168.3.4-192.168.3.6-11311-60792-6
1    192.168.3.4-192.168.3.6-11311-60794-6
2    192.168.3.4-192.168.3.6-11311-39922-6
3    192.168.3.4-192.168.3.6-11311-55266-6
4    192.168.3.6-192.168.3.7-43770-11111-6
Name: Flow ID, dtype: object


Src IP
Unique values: 6
0    192.168.3.6
1    192.168.3.6
2    192.168.3.6
3    192.168.3.6
4    192.168.3.7
Name: Src IP, dtype: object


Dst IP
Unique values: 9
0    192.168.3.4
1    192.168.3.4
2    192.168.3.4
3    192.168.3.4
4    192.168.3.6
Name: Dst IP, dtype: object


Protocol
Unique values: 3
0    6
1    6
2    6
3    6
4    6
Name: Protocol, dtype: int64


Timestamp
Unique values: 13625
0    07/07/2023 02:10:23 PM
1    07/07/2023 02:10:23 PM
2    07/07/2023 02:10:32 PM
3    07/07/2023 02:11:11 PM
4    07/07/2023 02:10:03 PM
Name: Timestamp, dtype: object


Src Port
Unique values: 36374
0    60792
1    60794
2    39922
3    55266
4    11111
Name: Src Port, dtype: int64


Dst Port
Unique values: 13524
0  

In [12]:
ros = datasets["ROSIDS23"]

for col in [
    "Src IP",
    "Dst IP",
    "Protocol"
]:

    print("\n================")
    print(col)

    print(
        pd.crosstab(
            ros[col],
            ros["Label"],
            normalize='index'
        ).round(3)
    )


Src IP
Label         Benign    DoS  Subflood  UnauthPub  UnauthSub
Src IP                                                     
192.168.3.10   0.092  0.444     0.417      0.003      0.043
192.168.3.14   0.909  0.000     0.000      0.000      0.091
192.168.3.4    0.839  0.004     0.017      0.114      0.026
192.168.3.6    0.821  0.000     0.017      0.116      0.046
192.168.3.7    0.955  0.001     0.011      0.018      0.015
8.6.0.1        0.949  0.003     0.007      0.020      0.020

Dst IP
Label            Benign    DoS  Subflood  UnauthPub  UnauthSub
Dst IP                                                        
192.168.3.10      0.411  0.055     0.000      0.116      0.419
192.168.3.4       0.294  0.392     0.259      0.038      0.017
192.168.3.6       0.684  0.000     0.183      0.084      0.049
192.168.3.7       0.940  0.001     0.018      0.018      0.023
224.0.0.251       0.960  0.000     0.000      0.000      0.040
224.0.0.7         0.947  0.000     0.017      0.013      0.023


In [13]:
for col in [
    "Src Port",
    "Dst Port"
]:

    print("\n================")
    print(col)

    print(
        ros.groupby("Label")[col].nunique()
    )


Src Port
Label
Benign       13921
DoS          24995
Subflood     15632
UnauthPub     3242
UnauthSub     3126
Name: Src Port, dtype: int64

Dst Port
Label
Benign       12056
DoS            143
Subflood      5429
UnauthPub     3105
UnauthSub     1607
Name: Dst Port, dtype: int64


In [14]:
ros = datasets["ROSIDS23"].copy()

ros['Timestamp'] = pd.to_datetime(
    ros['Timestamp']
)

ros.groupby('Label')['Timestamp'].agg(
    ['min','max']
)

,min,max
Label,,
Benign,2023-07-07 14:10:03,2023-08-15 16:28:32
DoS,2023-07-12 17:05:33,2023-07-12 18:14:09
Subflood,2023-08-15 15:12:11,2023-08-15 16:08:20
UnauthPub,2023-07-12 10:40:50,2023-07-12 11:34:12
UnauthSub,2023-07-12 12:38:57,2023-07-12 13:32:55


In [15]:
ros.groupby(
    [
        ros['Timestamp'].dt.hour,
        'Label'
    ]
).size().unstack(fill_value=0)

Label,Benign,DoS,Subflood,UnauthPub,UnauthSub
Timestamp,,,,,
10,2760,0,0,2029,0
11,4214,0,0,5788,0
12,3029,0,0,0,1473
13,4673,0,0,0,3816
14,5175,0,0,0,0
15,16831,0,23426,0,0
16,11106,0,6638,0,0
17,10556,24104,0,0,0
18,4167,6896,0,0,0


# **Dataset Quality Analysis**

In [16]:
ros = datasets["ROSIDS23"]

missing = ros.isnull().sum()
print("Missing value:")
print(missing[missing > 0])

print("\n")
# np.isinf(
#     ros.select_dtypes(include=np.number)
# ).sum().sort_values(ascending=False).head(20)
print("Infinite Value:")
print(np.isinf(
    ros.select_dtypes(include=np.number)
).sum().sort_values(ascending=False).head(20))

print("\n")
print("Duplicate Value:")
ros.duplicated().sum()

Missing value:
Flow Byts/s    272
dtype: int64


Infinite Value:
Flow Pkts/s         275
Flow Byts/s           3
Src Port              0
Flow Duration         0
Tot Fwd Pkts          0
Dst Port              0
Protocol              0
TotLen Bwd Pkts       0
Fwd Pkt Len Max       0
Fwd Pkt Len Min       0
Fwd Pkt Len Mean      0
Fwd Pkt Len Std       0
Bwd Pkt Len Max       0
Tot Bwd Pkts          0
TotLen Fwd Pkts       0
Bwd Pkt Len Mean      0
Bwd Pkt Len Min       0
Bwd Pkt Len Std       0
Flow IAT Mean         0
Flow IAT Std          0
dtype: int64


Duplicate Value:


np.int64(0)

In [17]:
ros = datasets["ROSIDS23"]

# Remove label temporarily
X = ros.drop("Label", axis=1)


# Calculate variance
variance = X.var(numeric_only=True)


# Find zero variance features
zero_variance_features = variance[
    variance == 0
]


print("Number of zero variance features:",
      len(zero_variance_features))


print("\nZero variance features:")
print(zero_variance_features)

Number of zero variance features: 14

Zero variance features:
Fwd PSH Flags        0.0
Fwd URG Flags        0.0
Bwd URG Flags        0.0
URG Flag Cnt         0.0
CWE Flag Count       0.0
ECE Flag Cnt         0.0
Fwd Byts/b Avg       0.0
Fwd Pkts/b Avg       0.0
Fwd Blk Rate Avg     0.0
Bwd Byts/b Avg       0.0
Bwd Pkts/b Avg       0.0
Bwd Blk Rate Avg     0.0
Init Fwd Win Byts    0.0
Fwd Seg Size Min     0.0
dtype: float64


In [18]:
nunique = X.nunique()

near_constant = nunique[
    nunique <= 2
]

print(
    "Features with <=2 unique values:",
    len(near_constant)
)

print(near_constant)

Features with <=2 unique values: 20
Fwd PSH Flags        1
Bwd PSH Flags        2
Fwd URG Flags        1
Bwd URG Flags        1
FIN Flag Cnt         2
SYN Flag Cnt         2
RST Flag Cnt         2
PSH Flag Cnt         2
ACK Flag Cnt         2
URG Flag Cnt         1
CWE Flag Count       1
ECE Flag Cnt         1
Fwd Byts/b Avg       1
Fwd Pkts/b Avg       1
Fwd Blk Rate Avg     1
Bwd Byts/b Avg       1
Bwd Pkts/b Avg       1
Bwd Blk Rate Avg     1
Init Fwd Win Byts    1
Fwd Seg Size Min     1
dtype: int64


# **Correlation Analysis**

In [19]:
ros = datasets["ROSIDS23"]

X = ros.drop("Label", axis=1)

numeric_X = X.select_dtypes(
    include=['int64','float64']
)

corr_matrix = numeric_X.corr()


# Find highly correlated pairs

high_corr = []

for i in range(len(corr_matrix.columns)):
    for j in range(i+1,len(corr_matrix.columns)):

        if abs(corr_matrix.iloc[i,j]) > 0.95:

            high_corr.append(
                (
                    corr_matrix.columns[i],
                    corr_matrix.columns[j],
                    corr_matrix.iloc[i,j]
                )
            )


high_corr_df = pd.DataFrame(
    high_corr,
    columns=[
        "Feature1",
        "Feature2",
        "Correlation"
    ]
)


high_corr_df

,Feature1,Feature2,Correlation
0,Protocol,Pkt Len Min,0.951946
1,Tot Fwd Pkts,Tot Bwd Pkts,0.979285
2,Tot Fwd Pkts,Fwd Header Len,0.998984
3,Tot Fwd Pkts,Bwd Header Len,0.979318
4,Tot Fwd Pkts,Subflow Fwd Pkts,1.000000
5,Tot Fwd Pkts,Subflow Bwd Pkts,0.979285
6,Tot Bwd Pkts,Fwd Header Len,0.980805
7,Tot Bwd Pkts,Bwd Header Len,0.999999
8,Tot Bwd Pkts,Subflow Fwd Pkts,0.979285
9,Tot Bwd Pkts,Subflow Bwd Pkts,1.000000


# **Data Preparation**

In [20]:
ros = datasets["ROSIDS23"].copy()

print("Original shape:")
print(ros.shape)

print("\nColumns:")
print(len(ros.columns))

Original shape:
(136681, 84)

Columns:
84
